In [ ]:
import os
import math
import json
import zipfile
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt


# ----------------------------
# ZIP HELPER
# ----------------------------
def zip_output_dir(output_dir, zip_path):
    output_dir = os.path.abspath(output_dir)
    zip_path = os.path.abspath(zip_path)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(output_dir):
            for name in files:
                full_path = os.path.join(root, name)
                arcname = os.path.relpath(full_path, os.path.dirname(output_dir))
                zf.write(full_path, arcname)

    return zip_path


# ----------------------------
# CONFIG
# ----------------------------
TOKENIZED_CORPUS_PATH = "/kaggle/input/processed-corpus-tokens-q1/tokens.txt"
VOCAB_FILE_PATH       = "/kaggle/input/vocab-q3/vocab_q3.txt"
WORKING_DIRECTORY     = "/kaggle/working/q3_out"

BATCH_SIZE = 1024
LEARNING_RATE = 1e-3
EPOCHS = 12
UNK_PROB = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIGS = [
    {"name": "ctx4_120", "ctx": 4, "emb": 66, "hid": 120},
    {"name": "ctx6_120", "ctx": 6, "emb": 66, "hid": 120},
    {"name": "ctx4_240", "ctx": 4, "emb": 66, "hid": 240},
]

def load_vocab(vocab_path: str):
    vocab = {}
    with open(vocab_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            tok = line.rstrip("\n")
            vocab[tok] = idx
    id_to_token = {i: t for t, i in vocab.items()}

    for sp in ["<s>", "</s>", "<UNK>"]:
        if sp not in vocab:
            raise ValueError(f"Missing special token in vocab: {sp}")

    return vocab, id_to_token

def read_tokenized_corpus_to_ids(tokenized_path: str, vocab: dict):
    s_id = vocab["<s>"]
    eos_id = vocab["</s>"]
    unk_id = vocab["<UNK>"]

    sentences = []
    with open(tokenized_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            toks = line.split()  # already WordPiece token strings
            ids = [s_id] + [vocab.get(t, unk_id) for t in toks] + [eos_id]
            sentences.append(ids)
    return sentences


def make_windows(sent_ids, context_size: int, vocab: dict):
    pad_id = vocab["<s>"]
    ctx_len = context_size - 1
    windows = []

    for sent in sent_ids:
        ctx = deque([pad_id] * ctx_len, maxlen=ctx_len)
        for tok in sent:
            windows.append((list(ctx), tok))
            ctx.append(tok)

    return windows


def split_train_val(windows, val_frac: float = 0.1):
    split_idx = int(len(windows) * (1 - val_frac))
    return windows[:split_idx], windows[split_idx:]


def windows_to_loader(windows, batch_size: int, shuffle: bool):
    contexts = torch.tensor([c for c, _ in windows], dtype=torch.long)
    targets  = torch.tensor([t for _, t in windows], dtype=torch.long)
    ds = TensorDataset(contexts, targets)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


class NPLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, context_size: int, hidden_dim: int):
        super().__init__()
        self.context_size = context_size
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.fc1 = nn.Linear((context_size - 1) * emb_dim, hidden_dim)
        self.act = nn.Tanh()
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x: (B, context_size-1)
        e = self.emb(x)                  # (B, ctx_len, emb_dim)
        flat = e.reshape(e.size(0), -1)  # (B, ctx_len*emb_dim)
        h = self.act(self.fc1(flat))     # (B, hidden_dim)
        logits = self.fc2(h)             # (B, vocab_size)
        return logits


# ----------------------------
# 5) METRICS (LOSS, PPL, ACC)
# ----------------------------
def compute_metrics(model, loader):
    """
    Returns:
      avg_ce_loss (mean cross-entropy per token),
      ppl,
      acc
    """
    model.eval()
    crit_sum = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for ctx, tgt in loader:
            ctx = ctx.to(DEVICE)
            tgt = tgt.to(DEVICE)

            logits = model(ctx)
            total_loss += crit_sum(logits, tgt).item()
            pred = logits.argmax(dim=1)
            correct += (pred == tgt).sum().item()
            total += tgt.size(0)

    avg_loss = total_loss / max(total, 1)
    ppl = math.exp(min(avg_loss, 50))
    acc = correct / max(total, 1)
    return avg_loss, ppl, acc


# ----------------------------
# 6) TRAIN (REPORT EPOCH-WISE METRICS)
# ----------------------------
def train_one_model(model, train_loader, val_loader, vocab, epochs: int, lr: float, unk_prob: float = 0.01):
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=lr)
    unk_id = vocab["<UNK>"]

    history = []  # list of dicts per epoch

    for ep in range(epochs):
        model.train()

        # ---- training loop (opt step) ----
        running = 0.0
        for ctx, tgt in tqdm(train_loader, desc=f"Epoch {ep+1}/{epochs}"):
            ctx = ctx.to(DEVICE)
            tgt = tgt.to(DEVICE)

            # Replace context tokens with <UNK> with probability 0.01
            if unk_prob > 0:
                mask = (torch.rand_like(ctx.float()) < unk_prob)
                ctx = ctx.masked_fill(mask, unk_id)

            opt.zero_grad()
            logits = model(ctx)
            loss = crit(logits, tgt)
            loss.backward()
            opt.step()
            running += loss.item()

        # avg batch loss (not exactly per-token, but good curve)
        train_loss_curve = running / max(len(train_loader), 1)

        # ---- compute real metrics (per-token CE, PPL, ACC) ----
        train_ce, train_ppl, train_acc = compute_metrics(model, train_loader)
        val_ce, val_ppl, val_acc = compute_metrics(model, val_loader)

        row = {
            "epoch": ep + 1,
            "train_loss_curve": float(train_loss_curve),
            "train_ce": float(train_ce),
            "train_ppl": float(train_ppl),
            "train_acc": float(train_acc),
            "val_ce": float(val_ce),
            "val_ppl": float(val_ppl),
            "val_acc": float(val_acc),
        }
        history.append(row)

        print(
            f"Epoch {ep+1}: "
            f"train_ce={train_ce:.4f} train_ppl={train_ppl:.2f} train_acc={train_acc:.4f} | "
            f"val_ce={val_ce:.4f} val_ppl={val_ppl:.2f} val_acc={val_acc:.4f}"
        )

    return history


# ----------------------------
# 7) WORDPIECE DETOKENIZER
# ----------------------------
def detokenize_wordpiece(tokens):
    out = []
    for tok in tokens:
        if tok == "<s>":
            continue
        if tok == "</s>":
            break
        if tok.startswith("##"):
            if out:
                out[-1] += tok[2:]
            else:
                out.append(tok[2:])
        else:
            out.append(tok)
    return " ".join(out)


# ----------------------------
# 8) GENERATION (SEED EXPECTED IN OPTION-A TOKENS)
# ----------------------------
def generate(model, seed_line_tokens: str, k: int, vocab: dict, id_to_token: dict, context_size: int):
    model.eval()
    unk_id = vocab["<UNK>"]
    pad_id = vocab["<s>"]
    eos_id = vocab["</s>"]
    ctx_len = context_size - 1

    seed_tokens = seed_line_tokens.strip().split()
    seed_ids = [vocab.get(t, unk_id) for t in seed_tokens]

    if len(seed_ids) < ctx_len:
        ctx = [pad_id] * (ctx_len - len(seed_ids)) + seed_ids
    else:
        ctx = seed_ids[-ctx_len:]

    gen_ids = []
    for _ in range(k):
        x = torch.tensor([ctx], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            logits = model(x)
        nxt = int(logits.argmax(dim=1).item())
        gen_ids.append(nxt)
        ctx = ctx[1:] + [nxt]
        if nxt == eos_id:
            break

    gen_tokens = [id_to_token.get(i, "<UNK>") for i in gen_ids]
    return detokenize_wordpiece(seed_tokens + gen_tokens)


# ----------------------------
# 9) CSV SAVE (NO PANDAS NEEDED)
# ----------------------------
def save_csv(rows, path):
    if not rows:
        return
    keys = list(rows[0].keys())
    with open(path, "w", encoding="utf-8") as f:
        f.write(",".join(keys) + "\n")
        for r in rows:
            f.write(",".join(str(r[k]) for k in keys) + "\n")


# ----------------------------
# MAIN
# ----------------------------
def main():
    os.makedirs(WORKING_DIRECTORY, exist_ok=True)
    print("Device:", DEVICE)

    vocab, id_to_token = load_vocab(VOCAB_FILE_PATH)
    print("Loaded vocab size:", len(vocab))

    sent_ids = read_tokenized_corpus_to_ids(TOKENIZED_CORPUS_PATH, vocab)
    print("Loaded tokenized sentences:", len(sent_ids))

    all_results = {}     # per model name: history + final metrics
    all_metrics_rows = []  # flat rows for CSV

    for cfg in CONFIGS:
        print("\n" + "=" * 90)
        print("Training config:", cfg)

        windows = make_windows(sent_ids, cfg["ctx"], vocab)
        train_w, val_w = split_train_val(windows, val_frac=0.1)

        train_loader = windows_to_loader(train_w, BATCH_SIZE, shuffle=True)
        val_loader   = windows_to_loader(val_w,   BATCH_SIZE, shuffle=False)

        model = NPLM(len(vocab), cfg["emb"], cfg["ctx"], cfg["hid"]).to(DEVICE)

        history = train_one_model(
            model, train_loader, val_loader, vocab,
            epochs=EPOCHS, lr=LEARNING_RATE, unk_prob=UNK_PROB
        )

        # final metrics = last epoch
        final = history[-1]

        # save model checkpoint
        save_path = os.path.join(WORKING_DIRECTORY, f"nplm_{cfg['name']}.pth")
        torch.save({
            "state_dict": model.state_dict(),
            "cfg": cfg,
            "vocab": vocab,
            "special_ids": {
                "s_id": vocab["<s>"],
                "eos_id": vocab["</s>"],
                "unk_id": vocab["<UNK>"],
            },
        }, save_path)
        print("Saved model:", save_path)

        all_results[cfg["name"]] = {
            "cfg": cfg,
            "history": history,
            "final": final,
        }

        # add to flat table
        for row in history:
            all_metrics_rows.append({"model": cfg["name"], **row})

    # ----------------------------
    # SAVE METRICS FILES
    # ----------------------------
    metrics_csv_path = os.path.join(WORKING_DIRECTORY, "metrics.csv")
    metrics_json_path = os.path.join(WORKING_DIRECTORY, "metrics.json")
    save_csv(all_metrics_rows, metrics_csv_path)
    with open(metrics_json_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    print("\nSaved metrics:")
    print(" -", metrics_csv_path)
    print(" -", metrics_json_path)

    # ----------------------------
    # PRINT FINAL SUMMARY TABLE
    # ----------------------------
    print("\nFINAL SUMMARY (last epoch)")
    print("=" * 90)
    print(f"{'Model':<10} | {'Train PPL':<10} | {'Val PPL':<10} | {'Train Acc':<10} | {'Val Acc':<10}")
    print("=" * 90)
    for name in [c["name"] for c in CONFIGS]:
        fin = all_results[name]["final"]
        print(
            f"{name:<10} | "
            f"{fin['train_ppl']:<10.2f} | {fin['val_ppl']:<10.2f} | "
            f"{fin['train_acc']:<10.4f} | {fin['val_acc']:<10.4f}"
        )
    print("=" * 90)

    # ----------------------------
    # PLOT LOSS CURVES (use train/val CE for curves)
    # ----------------------------
    loss_path = os.path.join(WORKING_DIRECTORY, "loss_curves.png")
    plt.figure(figsize=(12, 6))
    for cfg in CONFIGS:
        name = cfg["name"]
        hist = all_results[name]["history"]
        train_ce = [h["train_ce"] for h in hist]
        val_ce   = [h["val_ce"] for h in hist]
        plt.plot(train_ce, label=f"{name} train", linewidth=2)
        plt.plot(val_ce, linestyle="--", label=f"{name} val", linewidth=2)

    plt.title("Loss Curves")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(loss_path, dpi=150)
    plt.close()
    print("\nSaved:", loss_path)

  
    base_name = CONFIGS[0]["name"]
    base_ctx = CONFIGS[0]["ctx"]

    # Reload model object for generation (we kept it inside local variable earlier,
    # but easiest is to rebuild from checkpoint if needed. Here we just reuse training model.
    # We'll store it again by re-instantiating and loading:
    ckpt = torch.load(os.path.join(WORKING_DIRECTORY, f"nplm_{base_name}.pth"), map_location=DEVICE)
    base_model = NPLM(len(ckpt["vocab"]), ckpt["cfg"]["emb"], ckpt["cfg"]["ctx"], ckpt["cfg"]["hid"]).to(DEVICE)
    base_model.load_state_dict(ckpt["state_dict"])
    base_model.eval()

    seeds = [
        "भारत एक विशाल",
        "सरकार ने आज",
        "खेल के मैदान",
        "मुझे लगता है",
    ]

    outs = []
    for s in seeds:
        out = generate(base_model, s, k=15, vocab=vocab, id_to_token=id_to_token, context_size=base_ctx)
        outs.append(out)
        print("\nSeed:", s)
        print("Gen :", out)

    gen_path = os.path.join(WORKING_DIRECTORY, "generated_output.txt")
    with open(gen_path, "w", encoding="utf-8") as f:
        for o in outs:
            f.write(o + "\n")
    print("\nSaved:", gen_path)
    zip_path = "/kaggle/working/q3_out.zip"
    zip_output_dir(WORKING_DIRECTORY, zip_path)
    print("\nZipped outputs to:", zip_path)


if __name__ == "__main__":
    main()
